# 利用 Python 处理中文新闻数据集



## 3.1 数据集介绍



### 3.1.1 数据集基本信息

本次实验使用的是一个中文新闻文本数据集，具有以下特点：

- **数据规模**：总共 50000 条新闻文本
- **分类类别**：5 种类型
  - 科技
  - 社会
  - 娱乐
  - 财经
  - 体育
- **数据集路径**：`/home/jovyan/work/datasets/5fbcdfa05005208e83d1ede4-momodel/news`



### 3.1.2 查看数据集目录结构

首先，让我们查看数据集所在的目录：

In [ ]:
# 导入必要的库
import os

# 查看数据集目录
!ls /home/jovyan/work/datasets/5fbcdfa05005208e83d1ede4-momodel

### 3.1.3 读取数据集

接下来，我们将数据集读取到字典中，其中：

- **键（key）**：新闻文本内容
- **值（value）**：新闻类别（科技、社会、娱乐、财经、体育）

这种数据结构便于后续的文本分析和处理。

In [ ]:
# 创建空字典存储数据集
dataset = {}
dataset_path = "/home/jovyan/work/datasets/5fbcdfa05005208e83d1ede4-momodel/news"

# 获取目录下所有文件
files = os.listdir(dataset_path)

# 遍历每个文件
for file in files:
    path = os.path.join(dataset_path, file)
    # 跳过隐藏文件和文件夹
    if not os.path.isdir(path) and not file[0] == '.':
        # 打开文件，使用 UTF-8 编码读取中文
        f = open(path, 'r', encoding='UTF-8')
        # 读取每一行新闻
        for line in f.readlines():
            # 将新闻内容作为键，类别（去掉 .txt 后缀）作为值
            dataset[line] = file[:-4]
        f.close()

print(f"数据集加载完成！总共加载了 {len(dataset)} 条新闻。")

### 3.1.4 展示样本数据

现在让我们查看每个类别的一条新闻示例，以了解数据的内容和格式：

In [ ]:
# 定义新闻类别
types = ("科技", "社会", "娱乐", "财经", "体育")

# 显示每个类别的第一条新闻
for t in types:
    for k, v in list(dataset.items()):
        if v == t:
            print(f"类别：{v}")
            print(f"新闻内容：{k[:200]}...")  # 只显示前200个字符
            print("-" * 80)
            break

## 3.2 数据集预处理



### 3.2.1 中文分词的重要性

在文本挖掘中，分词是最基础也是最重要的预处理步骤。

**为什么需要分词？**

- **英文**：单词之间天然有空格分隔，但也需要处理特殊情况（如 "New York" 应作为一个词）
- **中文**：汉字之间没有空格，必须通过专门的算法进行分词



### 3.2.2 jieba 分词工具

jieba 是目前最流行的 Python 中文分词库，支持三种分词模式：

1. **精确模式**：试图将句子最精确地切开，适合文本分析
2. **全模式**：把句子中所有可以成词的词语都扫描出来，速度快但不能解决歧义
3. **搜索引擎模式**：在精确模式基础上，对长词再次切分，提高召回率，适合用于搜索引擎分词

更多方法参考：https://github.com/fxsjy/jieba

### 3.2.3 导入分词库并准备样本

In [ ]:
# 导入 jieba 分词库
import jieba as jb

# 获取一条新闻作为示例
article = next(iter(dataset))
print(f"示例新闻（前100字）：{article[:100]}...")

#### 3.2.3.1 精确模式分词

精确模式会尝试将句子最精确地切开，适合文本分析。

In [ ]:
# 使用精确模式分词（cut_all=False）
article_split_precise = " ".join(jb.cut(article, cut_all=False))

print("精确模式分词结果（前200字）：")
print(article_split_precise[:200])
print("\n分词数量：", len(article_split_precise.split()))

#### 3.2.3.2 全模式分词

全模式会扫描出所有可能成词的词语，速度快但会有冗余。

In [ ]:
# 使用全模式分词（cut_all=True）
article_split_all = " ".join(jb.cut(article, cut_all=True))

print("全模式分词结果（前200字）：")
print(article_split_all[:200])
print("\n分词数量：", len(article_split_all.split()))

#### 3.2.3.3  搜索引擎模式分词

搜索引擎模式在精确模式基础上，对长词再次切分，适合搜索引擎使用。

In [ ]:
# 使用搜索引擎模式分词
article_split_search = " ".join(jb.cut_for_search(article))

print("搜索引擎模式分词结果（前200字）：")
print(article_split_search[:200])
print("\n分词数量：", len(article_split_search.split()))

**三种模式对比**

通过上面的结果可以看到：

- **精确模式**：分词数量适中，结果最精准
- **全模式**：分词数量最多，包含所有可能的词
- **搜索引擎模式**：分词数量介于两者之间，兼顾精确性和召回率

## 3.3 使用 TF-IDF 算法提取关键词



### 3.3.1 什么是 TF-IDF？

TF-IDF（Term Frequency-Inverse Document Frequency，词频-逆文档频率）是一种常用的文本挖掘加权技术。

**核心思想：**

- 如果某个词在一篇文章中出现频率高（TF高）
- 同时在其他文章中很少出现（IDF高）
- 则认为这个词具有很好的类别区分能力

**公式理解：**

- **TF（词频）**：某个词在文档中出现的次数
- **IDF（逆文档频率）**：总文档数 / 包含该词的文档数
- **TF-IDF = TF × IDF**



### 3.3.2 提取每个类别的关键词

In [ ]:
# 将每个类别的所有新闻合并成一个大文本
str_full = {}
for t in types:
    str_full[t] = ""

# 遍历数据集，将同类别的新闻合并
for k, v in dataset.items():
    str_full[v] += k

print("数据合并完成！")
for t in types:
    print(f"{t} 类别文本长度：{len(str_full[t])} 字符")

### 3.2.3 提取并展示每个类别的 Top 5 关键词

In [ ]:
# 导入 jieba 的 analyse 模块
import jieba.analyse

# 使用 TF-IDF 算法提取每个类别的关键词
print("各类别 Top 5 关键词（按 TF-IDF 权重排序）：\n")
print("=" * 60)

for k, v in str_full.items():
    print(f"\n【{k}】类别关键词：")
    # extract_tags: 提取关键词
    # topK=5: 提取权重最高的5个词
    # withWeight=True: 同时返回权重值
    keywords = jieba.analyse.extract_tags(v, topK=5, withWeight=True)
    for i, (word, weight) in enumerate(keywords, 1):
        print(f"  {i}. {word:10s} (权重: {weight:.6f})")
    print("-" * 60)

### 3.3.4 问题分析：为什么关键词中出现很多数字？

观察上面的结果，我们会发现一个有趣的现象：**很多类别的关键词中都出现了数字，比如 "10"**。

#### 原因分析

这是因为：

**1. 新闻文本中数字出现频率很高**

- **日期**：10月、10日、10号
- **时间**：10点、10时
- **数量**：10万、10亿、10个、10名
- **排名**：前10名、前10位、10强
- **其他**：第10轮、10年、10倍

**2. TF-IDF 算法的局限性**

TF-IDF 算法只计算词频和文档频率，并不理解词的"语义"：

- 它不知道 "10" 是一个数字
- 它不知道 "10" 对区分新闻类别没有帮助
- 只要某个词出现频率高，就会被认为是"重要"的

**3. 这是真实 NLP 项目中的常见问题**

在实际应用中，我们需要进行**数据清洗**和**特征筛选**，过滤掉这些无意义的词。

### 3.3.5 优化方案：过滤无意义的词

为了得到更有意义的关键词，我们需要对提取结果进行优化。

#### 优化策略

**1. 过滤纯数字**
- 使用 Python 的 `str.isdigit()` 方法判断是否为纯数字
- 过滤掉 "10"、"8"、"12" 这样的词

**2. 提取更多候选词**
- 先提取 20 个关键词
- 过滤后保留前 10 个有意义的词
- 确保有足够的有效关键词

**3. 进一步优化建议**
- 可以添加**停用词表**（常见的无意义词，如"的"、"了"等）
- 可以过滤**单字词**（保留更有意义的多字词）
- 可以添加**自定义词典**（领域专业术语）

让我们来看看优化后的效果：

In [ ]:
# 优化版本：过滤纯数字，提取更有意义的关键词
print("优化后各类别 Top 10 关键词（已过滤纯数字）：\n")
print("=" * 60)

for k, v in str_full.items():
    print(f"\n【{k}】类别关键词：")
    
    # 先提取20个候选关键词
    all_keywords = jieba.analyse.extract_tags(v, topK=20, withWeight=True)
    
    # 过滤掉纯数字的关键词，保留前10个
    filtered_keywords = [(word, weight) for word, weight in all_keywords if not word.isdigit()][:10]
    
    # 显示过滤后的关键词
    for i, (word, weight) in enumerate(filtered_keywords, 1):
        print(f"  {i}. {word:10s} (权重: {weight:.6f})")
    print("-" * 60)

### 3.3.6 优化结果分析

通过对比优化前后的结果，我们可以发现：

#### 优化效果对比

**优化前的问题：**
- 关键词中包含数字
- 这些数字对区分新闻类别没有实际帮助
- 浪费了关键词位置，掩盖了真正重要的特征词

**优化后的改进：**
- 过滤掉了无意义的数字
- 关键词更能体现各类别的特征
- 更适合用于文本分类等实际应用

#### 各类别特征分析

从优化后的关键词可以更清晰地看出：

- **科技类**：关键词集中在"手机"、"网络"、"用户"、"像素"等技术相关词汇
- **社会类**：关键词包含"记者"、"警方"、"民警"、"男子"等社会新闻常用词
- **娱乐类**：关键词有"电影"、"新浪"、"微博"、"娱乐"等娱乐媒体相关词
- **财经类**：关键词涉及"基金"、"市场"、"期货"、"公司"等金融术语
- **体育类**：关键词包括"比赛"、"球员"、"球队"、"赛季"等体育赛事用词

#### 实践启示

这个优化过程体现了 NLP 项目中的重要环节：

1. **数据清洗**：去除噪声数据，保留有价值的信息
2. **特征工程**：选择真正有区分度的特征
3. **迭代优化**：发现问题 → 分析原因 → 优化方案 → 验证效果

在实际项目中，我们可能还需要：
- 添加停用词过滤
- 使用词性标注，只保留名词、动词等
- 根据领域知识自定义词典
- 结合其他特征提取方法（如 Word2Vec）